# Design_Matrix Basics

This tutorial introduces the `Design_Matrix` class for creating and manipulating experimental design matrices.

In [ ]:
import numpy as np
from nltools.data import Design_Matrix
import matplotlib.pyplot as plt

## Creating a Design Matrix

Design matrices represent your experimental design with time on rows and conditions/regressors in columns.

In [ ]:
# Create a simple design matrix
# 100 time points, 2 Hz sampling rate
dm = Design_Matrix(np.zeros((100, 1)), sampling_freq=2)
print(f"Shape: {dm.shape}")
print(f"Sampling frequency: {dm.sampling_freq} Hz")

In [ ]:
# Add experimental conditions
# Condition A: occurs at 10s and 50s
# Condition B: occurs at 30s and 70s
dm["ConditionA"] = 0
dm.loc[10:15, "ConditionA"] = 1  # 5 second events
dm.loc[50:55, "ConditionA"] = 1

dm["ConditionB"] = 0
dm.loc[30:35, "ConditionB"] = 1
dm.loc[70:75, "ConditionB"] = 1

# Remove the initial zero column
dm = dm.drop(columns=[0])

## HRF Convolution

For fMRI studies, convolve stimulus events with the hemodynamic response function (HRF):

In [ ]:
# Convolve with HRF
dm_conv = dm.convolve()
print(f"Original max value: {dm.max().max():.2f}")
print(f"Convolved max value: {dm_conv.max().max():.2f}")

In [ ]:
# Plot to see the difference
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Original
ax1.plot(dm["ConditionA"], label="Condition A")
ax1.plot(dm["ConditionB"], label="Condition B")
ax1.set_ylabel("Original")
ax1.legend()
ax1.set_title("Original Design")

# Convolved
ax2.plot(dm_conv["ConditionA"], label="Condition A")
ax2.plot(dm_conv["ConditionB"], label="Condition B")
ax2.set_ylabel("HRF Convolved")
ax2.set_xlabel("Time (s)")
ax2.legend()
ax2.set_title("After HRF Convolution")

plt.tight_layout()
plt.show()

## Adding Nuisance Regressors

Add polynomial trends to model low-frequency drift:

In [ ]:
# Add polynomial regressors (linear and quadratic trends)
dm_with_poly = dm_conv.add_poly(order=2)
print(f"Columns: {dm_with_poly.columns.tolist()}")

In [ ]:
# Add an intercept
dm_with_poly["intercept"] = 1
print(f"Final design shape: {dm_with_poly.shape}")

## Visualization

Design_Matrix has built-in plotting methods:

In [ ]:
# Heatmap visualization
dm_with_poly.heatmap()

In [ ]:
# Check correlations between regressors
dm_with_poly.plot_corr()

## Working with Multiple Runs

Concatenate design matrices from multiple runs:

In [ ]:
# Create a second run with different timing
dm_run2 = Design_Matrix(np.zeros((100, 1)), sampling_freq=2)
dm_run2["ConditionA"] = 0
dm_run2.loc[20:25, "ConditionA"] = 1
dm_run2.loc[60:65, "ConditionA"] = 1

dm_run2["ConditionB"] = 0
dm_run2.loc[40:45, "ConditionB"] = 1
dm_run2.loc[80:85, "ConditionB"] = 1

dm_run2 = dm_run2.drop(columns=[0]).convolve()

# Append runs
dm_multi = dm_conv.append(dm_run2, unique_cols=["ConditionA", "ConditionB"])
print(f"Multi-run shape: {dm_multi.shape}")
print(f"Columns: {dm_multi.columns.tolist()}")

## Summary

- Design_Matrix extends pandas DataFrame for experimental designs
- Handles time-series data with specified sampling frequency
- Provides HRF convolution for fMRI designs
- Easy addition of nuisance regressors (polynomials, etc.)
- Built-in visualization and diagnostic tools
- Supports multi-run concatenation with proper handling